In [ ]:
from pynq import Overlay

def load_overlay(filename):
    # load overlay
    pl = Overlay(filename)
    # initialize coprocessor
    coproc = pl.search_0
    dma = pl.axi_dma_0
    return pl, coproc, dma

def get_degree(coproc):
    # get the degree of parallelization
    coproc.register_map.base_prn.base_prn = 0
    coproc.register_map.CTRL.AP_START = 1
    while coproc.register_map.CTRL.AP_DONE == 0:
        pass
    
    par_prn  = coproc.register_map.par_prn.par_prn
    par_freq = coproc.register_map.par_freq.par_freq
    par_code = coproc.register_map.par_code.par_code
    return par_prn, par_freq, par_code
    

def search_hw(coproc, dma, sig, base_prn, base_freq, base_code, corr, par_code):
    # invoke the coprocessor
    coproc.register_map.base_prn.base_prn = base_prn
    coproc.register_map.base_freq.base_freq = base_freq
    coproc.register_map.base_code.base_code = base_code
    coproc.register_map.corr_1.corr = corr.device_address
    coproc.register_map.CTRL.AP_START = 1
    dma.sendchannel.transfer(sig, base_code * 8, 16362 + par_code * 8)
    while coproc.register_map.CTRL.AP_DONE == 0:
        pass
    dma.sendchannel.wait()

In [2]:
# set up buffers and load input data
from pynq import allocate
import numpy as np
from pynq.overlays.base import BaseOverlay
base_overlay = BaseOverlay("base.bit")

SAMP_IN_MS = 16368 # number of samples per 1 code (1 ms)

if 'corr' not in globals() or corr is None:
    corr = allocate(shape=(33, 25, 2046), dtype=np.int32) # output buffer
if 'sig' not in globals() or sig is None:
    sig = allocate(shape=(SAMP_IN_MS*3,), dtype=np.uint8) # input buffer

inp = open("gps_64k.bin", "rb")
sig[:] = np.fromfile(inp, dtype=np.uint8, count=SAMP_IN_MS*3)
inp.close()

In [ ]:
# main process
from time import perf_counter

def main_hw(filename):
    PRN_MIN     = 1                 # satelite ID
    PRN_MAX     = 32
    FREQ_MIN    = -12
    FREQ_MAX    = 12
    
    pl, coproc, dma = load_overlay(filename)
    par_prn, par_freq, par_code = get_degree(coproc)
    
    start_time = perf_counter()
    
    for prn_idx in range(PRN_MIN, PRN_MAX + 1, par_prn):
        for freq_idx in range(FREQ_MIN, FREQ_MAX + 1, par_freq):
            for code_idx in range(0, 2046, par_code):
                search_hw(coproc, dma, sig, prn_idx, freq_idx - FREQ_MIN, code_idx, corr, par_code)
    
    end_time = perf_counter()
    return end_time - start_time
        

In [ ]:
from glob import glob
for filename in glob("bits/*.bit"):
    elapsed_time = main_hw(filename))
    checksum = np.sum(corr[8:10,10:15,100:110])
    print("%s\t%.3f\t%d" % (filename, elapsed_time, checksum))

In [ ]:
# print the results of PRN 9 (for debug purpose)
NOISE_FLOOR = 4500000
np.set_printoptions(suppress=True)
np.array2string(corr[9] / NOISE_FLOOR, precision=2, separator=',', max_line_width=np.inf, threshold=np.inf, floatmode='fixed')